# Test: Does Sonnet batch parallel tool calls?

Strands' `ConcurrentToolExecutor` runs all `tool_use` blocks in one response turn concurrently (asyncio tasks). The question is: will Claude 3.5 Sonnet return all 3 specialist tool calls **in a single turn**, or one at a time?

This notebook uses 3 trivial mock tools (no paper loading, ~instant) to check the batching behavior cheaply before wiring up the real sub-agents.

In [3]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from strands import Agent, tool
from strands.models import BedrockModel
from core.settings import settings

# Confirmed ACTIVE cross-region inference profile (us-east-1, 2026-06-10)
SONNET_MODEL = 'us.anthropic.claude-sonnet-4-6'
print('Settings loaded. Sonnet model:', SONNET_MODEL)

Settings loaded. Sonnet model: us.anthropic.claude-sonnet-4-6


## Step 1 — Define 3 trivial mock tools

These mirror the real specialist tools but just return a fixed string instantly.
We're testing the orchestrator's *calling pattern*, not the specialist's *work*.

In [4]:
@tool
def specialist_attention(task: str) -> str:
    """Summarize how attention / transformer architecture handles long context.

    Args:
        task: The research comparison task.
    """
    return "MOCK: Attention specialist result — transformers use self-attention."


@tool
def specialist_position(task: str) -> str:
    """Summarize how input position affects long-context performance.

    Args:
        task: The research comparison task.
    """
    return "MOCK: Position specialist result — early/late tokens are recalled better."


@tool
def specialist_retrieval(task: str) -> str:
    """Summarize how retrieval augmentation addresses long-context limits.

    Args:
        task: The research comparison task.
    """
    return "MOCK: Retrieval specialist result — RAG avoids stuffing the context window."

print('3 mock tools defined.')

3 mock tools defined.


## Step 2 — Build the orchestrator

System prompt explicitly tells Sonnet to call all 3 tools. We'll check whether it returns all 3 `toolUse` blocks in ONE turn.

In [6]:
model = BedrockModel(
    model_id=SONNET_MODEL,
    region_name=settings.aws_region,
    temperature=0.0,
    max_tokens=1500,
)

SYSTEM = (
    "You are a research coordinator with three independent specialist tools: "
    "specialist_attention, specialist_position, specialist_retrieval. "
    "They cover different topics and are completely independent — call ALL THREE "
    "in a single batch. Then write a one-paragraph synthesis of the three results."
)

orchestrator = Agent(
    model=model,
    tools=[specialist_attention, specialist_position, specialist_retrieval],
    system_prompt=SYSTEM,
    callback_handler=None,
)
print('Orchestrator built.',orchestrator)

Orchestrator built. <strands.agent.agent.Agent object at 0x7804f877bb10>


## Step 3 — Run it and capture the messages

In [7]:
TASK = (
    "Compare how three areas handle long-context problems: "
    "(1) attention/architecture, (2) position effects, (3) retrieval."
)

result = orchestrator(TASK)
print('Run complete.')

Run complete.


## Step 4 — Inspect: how many tool calls per assistant turn?

We scan every assistant message and count `toolUse` blocks. If Sonnet batches all 3 into one turn → parallel (free via ConcurrentToolExecutor). If it calls them one at a time → sequential.

In [8]:
print('=== Message-by-message tool call breakdown ===')
for i, msg in enumerate(orchestrator.messages):
    role = msg['role']
    content = msg.get('content', [])
    tool_uses = [b for b in content if isinstance(b, dict) and 'toolUse' in b]
    tool_results = [b for b in content if isinstance(b, dict) and 'toolResult' in b]
    if tool_uses:
        names = [b['toolUse']['name'] for b in tool_uses]
        print(f'Turn {i} [{role}]: {len(tool_uses)} tool call(s) → {names}')
    elif tool_results:
        print(f'Turn {i} [{role}]: {len(tool_results)} tool result(s)')
    else:
        text_len = sum(len(b.get('text','')) for b in content if isinstance(b, dict))
        print(f'Turn {i} [{role}]: text ({text_len} chars)')

=== Message-by-message tool call breakdown ===
Turn 0 [user]: text (118 chars)
Turn 1 [assistant]: 3 tool call(s) → ['specialist_attention', 'specialist_position', 'specialist_retrieval']
Turn 2 [user]: 3 tool result(s)
Turn 3 [assistant]: text (2627 chars)


In [9]:
# The key number: max tool calls in a single assistant turn
max_calls_in_one_turn = 0
for msg in orchestrator.messages:
    if msg['role'] == 'assistant':
        n = sum(1 for b in msg.get('content', []) if isinstance(b, dict) and 'toolUse' in b)
        max_calls_in_one_turn = max(max_calls_in_one_turn, n)

print(f'Max tool calls in one turn: {max_calls_in_one_turn}')
if max_calls_in_one_turn == 3:
    print('✅ Sonnet batched all 3 — ConcurrentToolExecutor will run them in PARALLEL')
elif max_calls_in_one_turn == 1:
    print('⚠️  Sonnet called tools one at a time — execution is SEQUENTIAL')
else:
    print(f'ℹ️  Sonnet batched {max_calls_in_one_turn} tools in one turn')

Max tool calls in one turn: 3
✅ Sonnet batched all 3 — ConcurrentToolExecutor will run them in PARALLEL


## Result interpretation

| Result | What it means | Action |
|--------|--------------|--------|
| max = 3 | Sonnet batches all 3 in one turn → Strands runs them concurrently | Proceed with real orchestrator — parallel is free |
| max = 1 | Sequential tool calls | Real orchestrator still works, just slower — accept the tradeoff |
| max = 0 | Sonnet answered without calling any tool | Prompt needs to be stronger or model not available |